# Patrones de lectura: hold-out vs submission

Pregunta: ¿los lectores se quedan en los mismos **autores / géneros / editoriales** entre train y test? ¿Las 20 recomendaciones de `submission.csv` imitan ese comportamiento?

Protocolo (el mismo que `main.py`): `leave_last_k` con k=20, usuarios con ≥21 interacciones. En submission se compara contra el **historial completo** (como en la inferencia final), solo lectores con historial.

Definición de *lealtad*: para cada fila de test (o de submission), el valor de la dimensión ¿aparece al menos una vez en el historial previo de ese lector?

- **micro**: promedio sobre interacciones
- **macro**: promedio de tasas por usuario

Para regenerar tablas sin abrir el notebook: `uv run python analisis_patrones.py`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from quelibroleo.data import load_catalog, load_interactions
from quelibroleo.split import leave_last_k

ROOT = Path(".")
df = load_interactions(ROOT / "data.db")
libros, _lectores = load_catalog(ROOT / "data.db")
train, test = leave_last_k(df, k=20)
print(f"Train {train.shape}  Test {test.shape}  usuarios={test['id_lector'].nunique()}")

In [ ]:
def sets(part, col):
    return part.dropna(subset=[col]).groupby("id_lector")[col].agg(set)


def hit_rate(part, train_sets, col):
    mapped = part["id_lector"].map(train_sets)
    present = mapped.notna()
    hits = pd.Series(False, index=part.index)
    hits.loc[present] = [
        pd.notna(v) and v in s for v, s in zip(part.loc[present, col], mapped[present], strict=True)
    ]
    return hits


def summarize(part, history, dims, fase):
    rows = []
    for name, col in dims.items():
        tsets = sets(history, col)
        hits = hit_rate(part, tsets, col)
        urates = part.assign(_h=hits.to_numpy()).groupby("id_lector")["_h"].mean()
        rows.append(
            {
                "fase": fase,
                "dimension": name,
                "micro": float(hits.mean()),
                "macro": float(urates.mean()),
                "mediana": float(urates.median()),
                "pct_users_todo_1": float((urates == 1).mean()),
                "pct_users_nada_0": float((urates == 0).mean()),
            }
        )
    return pd.DataFrame(rows)


DIMS = {"autor": "autor", "genero": "genero", "editorial": "editorial", "id_libro": "id_libro"}
holdout = summarize(test, train, DIMS, "test_vs_train")
holdout

## Qué hacen los lectores en el hold-out

Resultados de una corrida (`analisis_patrones.py`):

| dimensión | micro test | azar (catálogo interactuado) |
|---|---:|---:|
| género | **0.832** | 0.623 |
| editorial | **0.587** | 0.285 |
| autor | **0.248** | 0.017 |
| mismo libro | 0.000 | ~0.002 |

Lectura:

1. **Género es el patrón fuerte.** El 83% de los libros del test están en un género que el usuario ya leyó en train. El 32% de los usuarios se queda *100%* dentro de géneros ya vistos. Un baseline de azar ya da 62% (pocos géneros populares), pero el comportamiento real está claramente por encima.
2. **Autor es lealtad real pero no dominante.** 25% vs 1.7% de azar: sí vuelven a autores conocidos, pero el autor favorito del train solo cubre el **3.7%** del test. Mediana de autores distintos en train: 27; en las 20 lecturas de test: 16. Exploración de autores, no saga-only.
3. **Editorial** a mitad de camino (59% vs 29%).
4. **No releen el mismo `id_libro`** en las últimas 20 (útil: filtrar vistos no oculta relecturas en este split).
5. El género #1 del train cubre solo el **28%** del test: hay un género dominante, pero el resto se reparte.

In [ ]:
pool = df[["id_libro", "autor", "genero", "editorial"]].drop_duplicates("id_libro")
rng = np.random.default_rng(42)
idx = rng.integers(0, len(pool), size=len(test))
fake = test[["id_lector"]].copy()
for c in ["id_libro", "autor", "genero", "editorial"]:
    fake[c] = pool.iloc[idx][c].to_numpy()
azar = summarize(fake, train, DIMS, "azar_catalogo_interactuado")
pd.concat([holdout, azar], ignore_index=True)

In [ ]:
def top1_hit(part, history, col):
    top = (
        history.groupby(["id_lector", col]).size().reset_index(name="n")
        .sort_values(["id_lector", "n"], ascending=[True, False])
        .groupby("id_lector", sort=False).head(1)
        .set_index("id_lector")[col]
    )
    return float((part[col] == part["id_lector"].map(top)).mean())

print("test en género #1 train", top1_hit(test, train, "genero"))
print("test en autor #1 train", top1_hit(test, train, "autor"))
print("autores únicos med train/test", train.groupby("id_lector")["autor"].nunique().median(), test.groupby("id_lector")["autor"].nunique().median())
print("géneros únicos med train/test", train.groupby("id_lector")["genero"].nunique().median(), test.groupby("id_lector")["genero"].nunique().median())

## Submission: ¿sigue la misma lógica?

No. Las recomendaciones **no reproducen** la lealtad observada.

| dimensión | test vs train | submission vs historial |
|---|---:|---:|
| género | 0.832 | **0.605** (≈ azar 0.623) |
| editorial | 0.587 | **0.149** (peor que azar 0.285) |
| autor | 0.248 | **0.012** (peor que azar 0.017) |
| autor #1 | 0.037 | **0.000** |
| género #1 | 0.281 | **0.078** |

El 89% de los lectores con historial **no recibe ni un libro de un autor que ya hayan leído**. El ranking se parece más a un catálogo genérico de ítems raros que a un perfil.

Además el **mix de géneros se invierte** respecto a lo que la gente lee:

| género | share train | share submission |
|---|---:|---:|
| Novela negra, intriga, terror | 0.208 | 0.022 |
| Narrativa | 0.171 | 0.120 |
| Ensayo | 0.028 | **0.154** |
| Infantil y juvenil | 0.056 | **0.108** |
| Cómics | 0.019 | **0.105** |
| Biografías | 0.022 | **0.087** |
| Autoayuda | 0.001 | **0.060** |

Popularidad (nº de interacciones del libro): **mediana 75 en test vs 1 en submission**. El modelo empuja cola larga. Un solo título (*El defensa del imperio…*) entra en el top-20 de **~72%** de los lectores del ejemplo; 27 libros cubren la mitad de todas las filas de `submission.csv`.

In [ ]:
sub = pd.read_csv(ROOT / "submission.csv")
hist_users = set(df["id_lector"].unique())
print("lectores submission", sub["id_lector"].nunique(), "sin historial", (~sub["id_lector"].drop_duplicates().isin(hist_users)).sum())

sub_join = sub.merge(libros[["id_libro", "autor", "genero", "editorial", "titulo"]], on="id_libro", how="left")
sub_hist = sub_join[sub_join["id_lector"].isin(hist_users)].copy()
sub_sum = summarize(sub_hist, df, DIMS, "submission_vs_historial")
pd.concat([holdout, sub_sum], ignore_index=True)

In [ ]:
pop = df.groupby("id_libro").size()
print("popularidad mediana test / sub", float(test["id_libro"].map(pop).median()), float(sub_hist["id_libro"].map(pop).median()))
print("géneros únicos med /20 recs", float(sub_hist.groupby("id_lector")["genero"].nunique().median()))
print("autores únicos med /20 recs", float(sub_hist.groupby("id_lector")["autor"].nunique().median()))

top_rec = (
    sub_join.groupby(["id_libro", "titulo", "autor", "genero"], dropna=False)
    .size().sort_values(ascending=False).head(12).reset_index(name="n_recs")
)
top_rec

## Implicación para el modelo

LightGBM predice *rating* y el ranking toma los scores más altos del catálogo. Ítems con `item_count` muy bajo (a menudo 1) pueden obtener scores altos y **el mismo bloque de libros oscuros se ofrece a casi todos**. Eso explica NDCG@20 ≈ 0: el test sigue novela negra / narrativa de catálogo popular; la submission no.

Direcciones coherentes con estos números (no implementadas acá):

- features o un término de ranking que premie **género ya leído** (el patrón más estable);
- penalizar o filtrar ítems de cola extrema (`item_count` mínimo);
- un modelo de ranking (no solo RMSE) o un blend con popularidad / ítems co-leídos;
- no esperar que “repetir autor” alcance: los usuarios exploran autores; el hueco está en género y en popularidad.